In [1]:
import datetime
import pandas as pd
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from os import listdir
from scipy.interpolate import RegularGridInterpolator
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm_notebook

In [5]:
das1=nc.Dataset("U&V&SLP150_180.nc") # type: ignore
das2=nc.Dataset("U&V&SLP180_250.nc") # type: ignore
lon1=np.array(das1['longitude'])
lon2=np.array(das2['longitude'])
lat=np.array(das1['latitude'])
lon=np.concatenate([lon1[:-1],lon2+360])
del lon1,lon2

In [6]:
print(das1.variables.keys())

dict_keys(['longitude', 'latitude', 'time', 'u10', 'v10', 'msl'])


In [13]:
u10_1=np.array(das1['u10']);u10_2=np.array(das2['u10'])
u10=np.concatenate([u10_1[:,:,:-1],u10_2],axis=-1)
del u10_1,u10_2

v10_1=np.array(das1['v10']);v10_2=np.array(das2['v10'])
v10=np.concatenate([v10_1[:,:,:-1],v10_2],axis=-1)
del v10_1,v10_2

msl_1=np.array(das1['msl']);msl_2=np.array(das2['msl'])
msl=np.concatenate([msl_1[:,:,:-1],msl_2],axis=-1)
del msl_1,msl_2

In [18]:
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))

times=pd.to_datetime(list(map(cftime2pdtime,nc.num2date(das1['time'][:],das1['time'].units))))


In [20]:
np.savez('SLP&U10&V10.npz',lon=lon,lat=lat,time=times,slp=msl,u10=u10,v10=v10)